In [2]:
# ============================================================
# CELL 1: PHYSICS CORE (this is the only "quantum" part)
# Manuscript convention:
#   |0> = U†|+z>, |1> = U†|-z>
#   P_theo(k) = |<k|+z>|^2
# Everything else in the notebook is just UI/packaging.
# ============================================================

import numpy as np

def u3(theta, phi, lam):
    """Single-qubit U3/ZYZ unitary used in manuscript (global phase irrelevant)."""
    c = np.cos(theta/2.0); s = np.sin(theta/2.0)
    return np.array([[c, -np.exp(1j*lam)*s],
                     [np.exp(1j*phi)*s, np.exp(1j*(phi+lam))*c]], dtype=complex)

def fix_global_phase(ket, tol=1e-14):
    """
    Stabilize displayed phases: make the largest-magnitude component real-positive.
    Avoids confusing sign/phase flips at angle boundaries.
    """
    ket = np.asarray(ket, dtype=complex)
    idx = int(np.argmax(np.abs(ket)))
    if np.abs(ket[idx]) < tol:
        return ket
    phase = np.exp(-1j*np.angle(ket[idx]))
    ket2 = phase * ket
    if np.real(ket2[idx]) < 0:
        ket2 = -ket2
    return ket2

def analyzer_basis_from_U(U):
    """
    Analyzer basis states in {|+z>, |-z>} coordinates:
      |0> = U†|+z> = first column of U†
      |1> = U†|-z> = second column of U†
    """
    Ud = U.conj().T
    ket0 = fix_global_phase(Ud[:, 0])
    ket1 = fix_global_phase(Ud[:, 1])
    return ket0, ket1

def probs_from_plus_z(ket0, ket1):
    """
    Fixed input |+z> = (1,0)^T.
    P_theo(k) = |<k|+z>|^2 = |(first component of |k>)|^2.
    """
    p0 = float(np.abs(ket0[0])**2)
    p1 = float(np.abs(ket1[0])**2)
    s = p0 + p1
    if s != 0:
        p0, p1 = p0/s, p1/s
    return np.array([p0, p1], dtype=float)

def sample_counts(probs, Ntrial, seed):
    """Multinomial sampling for empirical frequencies."""
    rng = np.random.default_rng(seed)
    counts = rng.multinomial(int(Ntrial), probs)
    freqs = counts / counts.sum()
    return counts, freqs


In [3]:
# ============================================================
# CELL 2: UI / DISPLAY (packaging; not required for physics understanding)
# Keeps your layout & output structure the same.
# ============================================================

from datetime import date
import matplotlib.pyplot as plt
from ipywidgets import (
    FloatSlider, FloatText, IntSlider, Dropdown, Button,
    HBox, VBox, Output, Layout
)
from IPython.display import display, clear_output, Markdown, HTML

# ------------------ Display / figure tuning ------------------
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (6.6, 4.0)
plt.rcParams['figure.autolayout'] = True

VERSION = "v1.3"
DATE = "02/17/2026"

def cfmt(z, sig=2):
    a, b = np.real(z), np.imag(z)
    return f"{'+' if a>=0 else '-'}{abs(a):.{sig}f}{'+' if b>=0 else '-'}{abs(b):.{sig}f}i"

# ------------------ Responsive widths ------------------
APP_MAX_W = '950px'
LEFT_W    = '300px'
COL_W     = '240px'
SLIM_W    = '200px'
DEG_W     = '120px'
LBL_W     = '28px'

# ------------------ Header ------------------
display(HTML(
    f"""
    <div style='width:100%; text-align:center; margin:16px 0;'>
      <span style='font-size:20px;'><b>Quantum Superposition Simulator (Version: {VERSION} | Date: {DATE})</b></span>
    </div>
    """
))
display(Markdown("Select U from Preset and press \"Apply\" to update."))
display(Markdown("If you enter numbers into fields, press \"return/enter\" before leaving the field to update."))

# ------------------ Widgets ------------------
preset = Dropdown(
    description='Preset',
    options=[('Custom','custom'),('Identity (I)','I'),('σ_x','X'),('σ_y','Y'),('σ_z','Z')],
    value='custom',
    layout=Layout(width='100%')
)
preset.style={'description_width':'70px'}

apply_btn = Button(description='Apply', layout=Layout(width='96px'))

nruns = IntSlider(description='$N_{trial}$', min=1, max=200000, step=1, value=1000,
                  continuous_update=False, layout=Layout(width='100%'))
nruns.style={'description_width':'70px'}

seed  = IntSlider(description='Seed',  min=0, max=2**31-1, step=1, value=1234,
                  continuous_update=False, layout=Layout(width='100%'))
seed.style={'description_width':'70px'}

theta = FloatSlider(description='θ', min=0.0, max=np.pi, step=0.001, value=0.0,
                    readout_format='.4f', continuous_update=False,
                    layout=Layout(width=SLIM_W))
theta.style={'description_width': LBL_W}

phi = FloatSlider(description='φ', min=0.0, max=2*np.pi, step=0.001, value=0.0,
                  readout_format='.4f', continuous_update=False,
                  layout=Layout(width=SLIM_W))
phi.style={'description_width': LBL_W}

lam = FloatSlider(description='λ', min=0.0, max=2*np.pi, step=0.001, value=0.0,
                  readout_format='.4f', continuous_update=False,
                  layout=Layout(width=SLIM_W))
lam.style={'description_width': LBL_W}

theta_deg = FloatText(description='θ°', value=0.0, step=0.1, layout=Layout(width=DEG_W))
phi_deg   = FloatText(description='φ°', value=0.0, step=0.1, layout=Layout(width=DEG_W))
lam_deg   = FloatText(description='λ°', value=0.0, step=0.1, layout=Layout(width=DEG_W))
for box in (theta_deg, phi_deg, lam_deg):
    box.style={'description_width': LBL_W}

out_text = Output(layout=Layout(
    border='1px solid #ddd', padding='8px',
    width='auto', height='340px', overflow_y='auto',
    flex='1 1 0', min_width='0'
))
out_plot = Output(layout=Layout(
    border='1px solid #ddd', padding='8px',
    width='auto', height='340px',
    flex='1 1 0', min_width='0'
))

# ------------------ Presets ------------------
def set_preset(tag):
    if tag=='I':
        theta.value=0.0; phi.value=0.0; lam.value=0.0
    elif tag=='Z':
        theta.value=0.0; phi.value=0.0; lam.value=np.pi
    elif tag=='X':
        theta.value=np.pi; phi.value=0.0; lam.value=np.pi
    elif tag=='Y':
        theta.value=np.pi; phi.value=np.pi/2; lam.value=np.pi/2

apply_btn.on_click(lambda _:
    set_preset(preset.value) if preset.value!='custom' else None
)

# ------------------ Degree sync ------------------
def _sync_deg_from_sliders(_=None):
    theta_deg.value = float(f"{np.degrees(theta.value):.4f}")
    phi_deg.value   = float(f"{np.degrees(phi.value):.4f}")
    lam_deg.value   = float(f"{np.degrees(lam.value):.4f}")

def _coerce_angle(x, lo, hi):
    if np.isnan(x) or np.isinf(x): return lo
    return float(min(max(x, lo), hi))

def _sync_sliders_from_deg(ch):
    src = ch['owner']
    if src is theta_deg: theta.value = _coerce_angle(np.radians(theta_deg.value), 0, np.pi)
    elif src is phi_deg: phi.value   = _coerce_angle(np.radians(phi_deg.value),   0, 2*np.pi)
    elif src is lam_deg: lam.value   = _coerce_angle(np.radians(lam_deg.value),   0, 2*np.pi)

# ------------------ Render ------------------
def render(*_):
    th, ph, la = theta.value, phi.value, lam.value
    U = u3(th, ph, la)

    # physics core call
    ket0, ket1 = analyzer_basis_from_U(U)
    probs = probs_from_plus_z(ket0, ket1)

    counts, freqs = sample_counts(probs, nruns.value, seed.value)

    with out_text:
        clear_output(wait=True)

        display(HTML(f"<pre style='margin:0;'><b>Unitary</b></pre>"))
        display(HTML(
            f"<pre style='margin:0;'>U(θ={th:.2f} rad, φ={ph:.2f} rad, λ={la:.2f} rad) = </pre>"
        ))
        for r in range(2):
            print("  [ " + " , ".join(cfmt(U[r,c], sig=4) for c in range(2)) + " ]")

        print()
        display(HTML("<b>Measured states:</b>"))
        print("Index:   State vector |k> in {|+z>, |-z>} basis (rows):")
        vec0 = "[" + " , ".join(cfmt(ket0[r], sig=2) for r in range(2)) + "]"
        vec1 = "[" + " , ".join(cfmt(ket1[r], sig=2) for r in range(2)) + "]"
        print(f"    0: {vec0}")
        print(f"    1: {vec1}")

        print()
        display(HTML("<b>Probabilities (Theory):"))
        display(HTML(f"<pre style='margin:0;'> P<sub>theo</sub>(0)=|⟨+z|0⟩|² = {probs[0]:.4f}"))
        display(HTML(f"<pre style='margin:0;'> P<sub>theo</sub>(1)=|⟨+z|1⟩|² = {probs[1]:.4f}"))

        display(HTML(f"<pre style='margin:0;'><b>Probabilities (Sampling/Empirical):</b>"))
        display(HTML(f"<pre style='margin:0;'> N<sub>trial</sub> = {nruns.value}, Seed = {seed.value}"))
        display(HTML(f"<pre style='margin:0;'> P<sub>samp</sub>(0) = {freqs[0]:.4f} (counts = {int(counts[0])})"))
        display(HTML(f"<pre style='margin:0;'> P<sub>samp</sub>(1) = {freqs[1]:.4f} (counts = {int(counts[1])})"))

    with out_plot:
        clear_output(wait=True)
        labels = ['P(+z)','P(-z)','P(0)','P(1)']
        heights = [1.0, 0.0, freqs[0], freqs[1]]
        x = np.arange(len(labels))
        fig, ax = plt.subplots()
        ax.bar(x, heights)
        ax.set_xticks(x, labels)
        ax.set_ylim(0, 1.2)
        ax.set_ylabel("Probability")
        ax.set_title("Histogram: Input + Outcome Probabilities", y=1.02, pad=10)

        for i, lab in enumerate(labels):
            if lab in ('P(0)','P(1)'):
                idx = 0 if lab=='P(0)' else 1
                y0 = heights[i]
                ax.text(i, y0 + 0.12, rf"$P_{{\mathrm{{theo}}}}({idx})={probs[idx]:.4g}$", ha='center', va='bottom')
                ax.text(i, y0 + 0.07, rf"$P_{{\mathrm{{samp}}}}({idx})={freqs[idx]:.4g}$", ha='center', va='bottom')
                ax.text(i, y0 + 0.02, rf"$N={counts[idx]}$", ha='center', va='bottom')

        ax.grid(True, axis='y', alpha=0.3)
        fig.text(0.32, 0.01, "Input", ha='center', va='center', fontsize=12, fontweight='bold')
        fig.text(0.75, 0.01, "Output", ha='center', va='center', fontsize=12, fontweight='bold')
        plt.show()
        display(HTML(f"<pre style='margin:0;'> N<sub>trial</sub> = {nruns.value}, Seed = {seed.value}"))

# ------------------ Observers ------------------
for w in (theta, phi, lam, preset, nruns, seed):
    w.observe(render, names='value')

for s in (theta, phi, lam):
    s.observe(_sync_deg_from_sliders, names='value')
_sync_deg_from_sliders()

for b in (theta_deg, phi_deg, lam_deg):
    b.observe(_sync_sliders_from_deg, names='value')

# ------------------ Layout ------------------
left_controls = VBox(
    [HBox([preset, apply_btn], layout=Layout(gap='8px')),
     nruns, seed],
    layout=Layout(width=LEFT_W)
)

theta_col = VBox([theta, theta_deg], layout=Layout(width=COL_W))
phi_col   = VBox([phi,   phi_deg],   layout=Layout(width=COL_W))
lam_col   = VBox([lam,   lam_deg],   layout=Layout(width=COL_W))

angles_row = HBox([theta_col, phi_col, lam_col],
                  layout=Layout(gap='12px', width='auto'))

top_row = HBox(
    [left_controls, angles_row],
    layout=Layout(justify_content='flex-start',
                  align_items='flex-start', gap='16px', width='100%')
)

bottom_outputs = HBox(
    [out_text, out_plot],
    layout=Layout(gap='12px', align_items='stretch', width='100%')
)

app_root = VBox([top_row, bottom_outputs],
                layout=Layout(width='100%', max_width=APP_MAX_W))

render()
display(app_root)


Select U from Preset and press "Apply" to update.

If you enter numbers into fields, press "return/enter" before leaving the field to update.